# PosteriorBench quickstart

This notebook walks through the `posteriorbench` package:

1. **Browse** the benchmark's tasks.
2. **Inspect** one task as a method sees it.
3. **Evaluate** a sampler on the task and read its cost and its KL divergences from the behavior posterior.
4. **Take corpus hits**, which are charged to a ledger.

**Setup:** install dependencies with `uv sync` from the repository root and start Jupyter with `uv run --with jupyter jupyter lab` (Jupyter is not among the package's dependencies). The task table and the evaluation file ship with the package, so parts 1 to 3 download no data. Part 3 runs Qwen3.5-0.8B, so we recommend a GPU; it also runs on CPU, more slowly. Part 4 reads the pair's hit file: from the Hugging Face dataset `Transluce/PosteriorBench` on first use, or from a local copy of the dataset given as `pb.Benchmark(hits_source=...)`.

## 1. Browse the tasks

A prompt-behavior pair (a `Task` in the package) joins one of 20 WildChat prompts with one of seven behaviors, each checked exactly by a substring or a regular expression. For each prompt the dataset holds 50 million responses of Qwen3.5-0.8B. A pair's hits among them are exact samples of its behavior posterior, and their share is the estimate $\hat p(S)$. The 108 prompt-behavior pairs with at least 500 hits are scored, and 21 of them are the benchmark's tasks: the pairs with the lowest, median and highest $\hat p(S)$ for each behavior.

In [ ]:
import posteriorbench as pb

bench = pb.Benchmark()  # the task table and the evaluation file bundled with the package
tasks = bench.tasks()  # the benchmark's 21 tasks; bench.tasks("scored") and bench.tasks("all") give more
print(f"{len(bench.tasks('all'))} prompt-behavior pairs, {len(bench.tasks('scored'))} scored pairs, {len(tasks)} benchmark tasks\n")

for task in tasks:
    print(f"{task.task_id:32s} p̂(S) = {task.probability:.1e}")

## 2. Inspect one task as a method sees it

A `Task` holds what a method may use: the prompt, the behavior and its rule. Each scored pair also has 250 references (hits held out for scoring) and a fallback response (one fixed hit). They are in a separate evaluation file, which only the evaluation code reads.

In [ ]:
task = bench.task("wc_1853/goblin_mention")
print("Prompt:  ", task.prompt)
print("Behavior:", task.behavior, f"(rule: {task.rule})")
print(f"p̂(S) = {task.hit_count:,} / {task.num_responses:,} = {task.probability:.1e}")

## 3. Evaluate a sampler

A sampler draws responses and computes the probability $q(y)$ of any response $y$. Here we use the Instruction reference sampler: the original model with a handwritten instruction as its system prompt. Writing the instruction is not charged and the plain instruction is not tested, so its construction cost is 0 inferences (an empty ledger); each response costs one inference.

`bench.evaluate` draws 1,024 new responses from the sampler and labels them; their hit fraction is the native hit rate $\hat q(S)$. If $\hat q(S)$ is below 10%, the sampler fails and gets the scores of the task's fallback. Otherwise it computes $p(y)$ and $q(y)$ for the 250 references and for up to 128 of the sampler's hits, and estimates the capped forward KL (cap $C = 50$) and the reverse KL between the sampler's hits and the posterior.

In [ ]:
model = pb.OriginalModel.from_pretrained()  # float32
candidate = pb.instruction_sampler(model, task)
result = bench.evaluate(task, candidate, model, ledger=pb.Ledger(), seed=0, replicate=0)

print(f"native hit rate {result.native_hit_rate:.3f}, fallback used: {result.fallback}")
print(f"capped forward KL {result.capped_forward_kl:.2f}, reverse KL {result.reverse_kl:.2f}")
print(
    f"construction cost {result.construction_inferences:.0f} inferences, "
    f"{result.inferences_per_response:.0f} per response"
)

A sampler that fails the threshold gets the fallback's scores. They use the stored $p(a)$ of the fallback response, so they need no model:

In [ ]:
fallback = bench.fallback_result(task)
print(f"capped forward KL {fallback.capped_forward_kl:.2f}, reverse KL {fallback.reverse_kl:.2f}")

## 4. Take corpus hits

Every corpus hit of a pair except its references may be used to build a sampler. `bench.corpus(task, ledger)` reads them from the pair's hit file, which does not hold the pair's references, and charges each distinct hit $1/\hat p(S)$ inferences to the method's ledger, the expected number of original-model draws that rejection sampling spends per hit. By default it also leaves out hits identical to a reference.

The hit files are part of the dataset on the Hugging Face Hub; the next cell downloads the task's file.

In [ ]:
ledger = pb.Ledger()
hits = bench.corpus(task, ledger).take_first(3)
print(f"3 corpus hits cost {ledger.inferences:,.0f} inferences; the first one has {len(hits[0].token_ids)} tokens")

## Where to go next

- To evaluate your own method, subclass `pb.Sampler` and implement `sample` and `logprobs` (and `inferences_per_response` if a response costs other than one inference, at most two), and charge your method's construction compute to a `pb.Ledger`. [`examples/02_custom_sampler`](../02_custom_sampler) shows a complete one; [docs/cost_accounting.md](../../docs/cost_accounting.md) explains what to charge.
- To compare methods, average each method's scores over replicates within a task, then over tasks with equal weights. `posteriorbench aggregate` does this for a set of result files, separately for each method and setting; `--replicate` sets the replicate index of an evaluation.
- The same evaluation from the command line:

```bash
uv run posteriorbench list-tasks
uv run posteriorbench evaluate --task wc_1853/goblin_mention --sampler instruction \
    --replicate 0 --out result.json
uv run posteriorbench aggregate result.json
```